In [ ]:
import sys
import subprocess

required = ["torch", "transformers", "datasets", "scikit-learn"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *required])
print("Installed required packages.")

In [ ]:
import math
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.to(device)
model.eval()

print(f"Loaded embedding model: {model_name}")
print(f"Hidden size: {model.config.hidden_size}")

In [ ]:
max_examples = 128
dataset = load_dataset("glue", "mrpc", split=f"validation[:{max_examples}]")

print("Dataset split: glue/mrpc validation")
print(f"Using first {len(dataset)} examples for fast evaluation")
print("Example row:")
print(dataset[0])

In [ ]:
def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    masked_embeddings = last_hidden_state * mask
    summed = masked_embeddings.sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts

def encode_texts(texts, batch_size=32, max_length=128):
    all_embeddings = []
    for start_idx in range(0, len(texts), batch_size):
        batch_texts = texts[start_idx:start_idx + batch_size]
        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
            embeddings = mean_pool(outputs.last_hidden_state, inputs["attention_mask"])
            embeddings = F.normalize(embeddings, p=2, dim=1)
        all_embeddings.append(embeddings.cpu())
    return torch.cat(all_embeddings, dim=0)

batch_size = 32
labels = dataset["label"]
sentence1_embeddings = encode_texts(dataset["sentence1"], batch_size=batch_size)
sentence2_embeddings = encode_texts(dataset["sentence2"], batch_size=batch_size)

similarity_scores = F.cosine_similarity(sentence1_embeddings, sentence2_embeddings, dim=1).tolist()

# Fixed threshold for cosine similarity-based paraphrase classification
threshold = 0.80
predictions = [1 if score >= threshold else 0 for score in similarity_scores]
confidences = [max(score, 1.0 - score) for score in similarity_scores]

print(f"Encoded {len(sentence1_embeddings)} sentence1 examples and {len(sentence2_embeddings)} sentence2 examples.")
print(f"Applied fixed cosine similarity threshold: {threshold:.2f}")

In [ ]:
accuracy = accuracy_score(labels, predictions)
precision, recall, f1, support_binary = precision_recall_fscore_support(
    labels, predictions, average="binary", zero_division=0
)
cm = confusion_matrix(labels, predictions)
per_class_precision, per_class_recall, per_class_f1, per_class_support = precision_recall_fscore_support(
    labels, predictions, labels=[0, 1], average=None, zero_division=0
)

print("Evaluation metrics on fixed 128-example subset:")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1       : {f1:.4f}")
print("Confusion matrix:")
print(cm)

print("Per-class metrics:")
label_map = {0: "not_paraphrase", 1: "paraphrase"}
for idx, label_id in enumerate([0, 1]):
    print(
        f"class={label_id} ({label_map[label_id]}), "
        f"precision={per_class_precision[idx]:.4f}, "
        f"recall={per_class_recall[idx]:.4f}, "
        f"f1={per_class_f1[idx]:.4f}, "
        f"support={per_class_support[idx]}"
    )

print("Classification report:")
print(classification_report(labels, predictions, target_names=["not_paraphrase", "paraphrase"], zero_division=0))

print("Similarity score summary:")
print(f"min={min(similarity_scores):.4f}")
print(f"max={max(similarity_scores):.4f}")
print(f"mean={sum(similarity_scores)/len(similarity_scores):.4f}")
sorted_scores = sorted(similarity_scores)
mid = len(sorted_scores) // 2
median = sorted_scores[mid] if len(sorted_scores) % 2 == 1 else (sorted_scores[mid - 1] + sorted_scores[mid]) / 2
print(f"median={median:.4f}")

In [ ]:
false_positives = []
false_negatives = []
threshold_borderline_errors = []

for i in range(len(dataset)):
    row = dataset[i]
    score = similarity_scores[i]
    distance_to_threshold = abs(score - threshold)
    record = {
        "index": i,
        "true_label": labels[i],
        "pred_label": predictions[i],
        "similarity": score,
        "threshold": threshold,
        "distance_to_threshold": distance_to_threshold,
        "sentence1": row["sentence1"],
        "sentence2": row["sentence2"],
    }
    if labels[i] == 0 and predictions[i] == 1:
        false_positives.append(record)
    elif labels[i] == 1 and predictions[i] == 0:
        false_negatives.append(record)
        
for r in false_positives + false_negatives:
    threshold_borderline_errors.append(r)

threshold_borderline_errors = sorted(threshold_borderline_errors, key=lambda x: x["distance_to_threshold"])

print(f"False positives: {len(false_positives)}")
print(f"False negatives: {len(false_negatives)}")
print(f"Total threshold-based errors: {len(threshold_borderline_errors)}")

def show_errors(title, rows, limit=5):
    print(title)
    if not rows:
        print("None")
        return
    print("idx | true | pred | similarity | threshold | dist | sentence1 | sentence2")
    for r in rows[:limit]:
        s1 = r["sentence1"].replace("\n", " ")[:70]
        s2 = r["sentence2"].replace("\n", " ")[:70]
        print(
            f"{r['index']:>3} | {r['true_label']} | {r['pred_label']} | "
            f"{r['similarity']:.4f} | {r['threshold']:.2f} | {r['distance_to_threshold']:.4f} | {s1} | {s2}"
        )

show_errors("Compact false positive table (first 5)", false_positives, limit=5)
print("-" * 120)
show_errors("Compact false negative table (first 5)", false_negatives, limit=5)
print("-" * 120)
show_errors("Threshold-borderline errors (closest to threshold, first 10)", threshold_borderline_errors, limit=10)

In [ ]:
print("RESULT SUMMARY")
print(f"model={model_name}")
print("inference_style=separate_sentence_embeddings_with_cosine_threshold")
print("dataset_split=glue/mrpc validation[:128]")
print(f"device={device}")
print(f"num_examples={len(dataset)}")
print(f"threshold={threshold:.2f}")
print(f"accuracy={accuracy:.4f}")
print(f"precision={precision:.4f}")
print(f"recall={recall:.4f}")
print(f"f1={f1:.4f}")
print(f"confusion_matrix={cm.tolist()}")
print(f"support_not_paraphrase={int(per_class_support[0])}")
print(f"support_paraphrase={int(per_class_support[1])}")
print(f"similarity_min={min(similarity_scores):.4f}")
print(f"similarity_max={max(similarity_scores):.4f}")
print(f"similarity_mean={sum(similarity_scores)/len(similarity_scores):.4f}")
print(f"false_positives={len(false_positives)}")
print(f"false_negatives={len(false_negatives)}")
print(f"threshold_based_errors={len(threshold_borderline_errors)}")